In [1]:
!pip install mlxtend
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram

from mlxtend.frequent_patterns import apriori, association_rules

import datetime as dt

In [2]:
import os
os.makedirs("results/plots", exist_ok=True)
os.makedirs("results/outputs", exist_ok=True)


# ===============================
# Load Dataset (Both Sheets)
# ===============================

file_path = "online_retail_II.xlsx"

sheet1 = pd.read_excel(file_path, sheet_name=0)
sheet2 = pd.read_excel(file_path, sheet_name=1)

df = pd.concat([sheet1, sheet2])

print("Dataset Shape:", df.shape)

Dataset Shape: (1067371, 8)


In [3]:
# ===============================
# Data Cleaning
# ===============================

df = df.dropna(subset=["Customer ID"])

df = df[~df["Invoice"].astype(str).str.contains("C")]

df = df[df["Quantity"] > 0]

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df["TotalPrice"] = df["Quantity"] * df["Price"]

df["Customer ID"] = df["Customer ID"].astype(int)

print("Cleaned Dataset:", df.shape)

Cleaned Dataset: (805620, 9)


In [4]:
# ===============================
# Exploratory Data Analysis
# ===============================

country_sales = df.groupby("Country")["TotalPrice"].sum().sort_values(ascending=False)

plt.figure(figsize=(10,5))
country_sales.head(10).plot(kind="bar")
plt.title("Top Countries by Revenue")
plt.tight_layout()

plt.savefig("results/plots/top_countries.png")
plt.close()


In [5]:
df["Month"] = df["InvoiceDate"].dt.to_period("M")

monthly_sales = df.groupby("Month")["TotalPrice"].sum()

plt.figure(figsize=(10,5))
monthly_sales.plot()
plt.title("Monthly Sales Trend")
plt.tight_layout()

plt.savefig("results/plots/monthly_sales.png")
plt.close()

In [6]:
top_products = df.groupby("Description")["Quantity"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,5))
top_products.plot(kind="bar")
plt.title("Top Selling Products")
plt.tight_layout()

plt.savefig("results/plots/top_products.png")
plt.close()

In [7]:
# ===============================
# RFM Feature Engineering
# ===============================

snapshot_date = df["InvoiceDate"].max() + dt.timedelta(days=1)

rfm = df.groupby("Customer ID").agg({
    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
    "Invoice": "count",
    "TotalPrice": "sum"
})

rfm.columns = ["Recency","Frequency","Monetary"]

print(rfm.head())


             Recency  Frequency  Monetary
Customer ID                              
12346            326         34  77556.46
12347              2        253   5633.32
12348             75         51   2019.40
12349             19        175   4428.69
12350            310         17    334.40


In [8]:
# ===============================
# Feature Scaling
# ===============================

scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm)


In [9]:
# ===============================
# KMeans Clustering
# ===============================

wcss = []

for i in range(1,11):

    kmeans = KMeans(n_clusters=i, random_state=42)

    kmeans.fit(rfm_scaled)

    wcss.append(kmeans.inertia_)


plt.plot(range(1,11), wcss)
plt.title("Elbow Method")
plt.xlabel("Clusters")
plt.ylabel("WCSS")

plt.savefig("results/plots/elbow_method.png")
plt.close()


kmeans = KMeans(n_clusters=5, random_state=42)

rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

In [10]:
# PCA Visualization
# ===============================

pca = PCA(n_components=2)

pca_data = pca.fit_transform(rfm_scaled)

plt.figure(figsize=(8,6))

plt.scatter(
    pca_data[:,0],
    pca_data[:,1],
    c=rfm["Cluster"]
)

plt.title("Customer Segments")

plt.savefig("results/plots/customer_clusters.png")

plt.close()

In [11]:
# DBSCAN
# ===============================

db = DBSCAN(eps=0.5, min_samples=5)

rfm["DBSCAN_cluster"] = db.fit_predict(rfm_scaled)

In [12]:
# Hierarchical Clustering
# ===============================

linked = linkage(rfm_scaled, method="ward")

plt.figure(figsize=(10,6))

dendrogram(linked)

plt.title("Hierarchical Clustering")

plt.savefig("results/plots/hierarchical.png")

plt.close()


In [13]:
# Cluster Insights
# ===============================

cluster_summary = rfm.groupby("Cluster").mean()

print(cluster_summary)

cluster_summary.to_csv("results/outputs/cluster_summary.csv")

            Recency    Frequency       Monetary  DBSCAN_cluster
Cluster                                                        
0         72.898425   119.338403    2249.382779       -0.000543
1        469.712293    42.944215     756.747243       -0.001550
2          1.666667  7913.000000  106042.183333       -1.000000
3          5.250000  2346.250000  424585.907500       -1.000000
4         27.537549   894.122530   22377.637921       -0.177866


In [21]:
# ===============================
# Market Basket Analysis (FP-Growth Optimized)
# ===============================

from mlxtend.frequent_patterns import fpgrowth

print("Preparing Market Basket Analysis...")

# Step 1: Keep top 100 products only
top_products = df['Description'].value_counts().head(100).index

df_filtered = df[df['Description'].isin(top_products)]

# Step 2: Create basket matrix
basket = (
    df_filtered
    .groupby(["Invoice", "Description"])["Quantity"]
    .sum()
    .unstack()
)

# Step 3: Convert to binary matrix
basket = basket.notnull().astype(int)

print("Basket Shape:", basket.shape)


# ===============================
# FP-Growth Algorithm
# ===============================

frequent_itemsets = fpgrowth(
    basket,
    min_support=0.02,
    use_colnames=True
)

print("Frequent Itemsets:", frequent_itemsets.shape)


# ===============================
# Association Rules
# ===============================

rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1
)

rules = rules.sort_values(by="lift", ascending=False)

print("Top Rules:")
print(rules.head())


# Save results
rules.to_csv("results/outputs/association_rules.csv")

Preparing Market Basket Analysis...
Basket Shape: (29048, 100)


c:\Users\91955\Downloads\ecommerce_seg\.venv\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


Frequent Itemsets: (129, 2)
Top Rules:
                                      antecedents  \
46   frozenset({GREEN REGENCY TEACUP AND SAUCER})   
47  frozenset({ROSES REGENCY TEACUP AND SAUCER })   
45              frozenset({DOLLY GIRL LUNCH BOX})   
44               frozenset({SPACEBOY LUNCH BOX })   
57        frozenset({ALARM CLOCK BAKELIKE GREEN})   

                                      consequents  antecedent support  \
46  frozenset({ROSES REGENCY TEACUP AND SAUCER })            0.032119   
47   frozenset({GREEN REGENCY TEACUP AND SAUCER})            0.036354   
45               frozenset({SPACEBOY LUNCH BOX })            0.034391   
44              frozenset({DOLLY GIRL LUNCH BOX})            0.038144   
57         frozenset({ALARM CLOCK BAKELIKE RED })            0.035837   

    consequent support   support  confidence       lift  representativity  \
46            0.036354  0.025578    0.796356  21.905819               1.0   
47            0.032119  0.025578    0.703598  21.

In [22]:
# Recommendation Function
# ===============================

def recommend(product):

    rec = rules[
        rules["antecedents"].apply(lambda x: product in x)
    ]

    return rec[["antecedents","consequents","confidence","lift"]]


print(recommend("WHITE HANGING HEART T-LIGHT HOLDER"))

                                        antecedents  \
10  frozenset({WHITE HANGING HEART T-LIGHT HOLDER})   
31  frozenset({WHITE HANGING HEART T-LIGHT HOLDER})   
23  frozenset({WHITE HANGING HEART T-LIGHT HOLDER})   
42  frozenset({WHITE HANGING HEART T-LIGHT HOLDER})   
2   frozenset({WHITE HANGING HEART T-LIGHT HOLDER})   
1   frozenset({WHITE HANGING HEART T-LIGHT HOLDER})   

                                       consequents  confidence      lift  
10   frozenset({RED HANGING HEART T-LIGHT HOLDER})    0.235884  4.201075  
31  frozenset({WOODEN PICTURE FRAME WHITE FINISH})    0.135843  2.379954  
23        frozenset({WOODEN FRAME ANTIQUE WHITE })    0.137070  2.245697  
42              frozenset({HEART OF WICKER LARGE})    0.132979  2.216159  
2            frozenset({HOME BUILDING BLOCK WORD})    0.120704  1.914911  
1       frozenset({ASSORTED COLOUR BIRD ORNAMENT})    0.132570  1.452067  


In [23]:
# Save Segments

rfm.to_csv("results/outputs/customer_segments.csv")
